# Laboratorio #6 - Aprendizaje por Refuerzo
* Paula Barillas - 22764
* Gerardo Pineda - 22880
* Mónica Salvatierra - 22249
* Bianca Calderón - 22272

- Link del repositorio: https://github.com/alee2602/LAB6-RL





**Contexto:**

Una empresa de robótica de asistencia médica está desarrollando un exoesqueleto para rehabilitación de movilidad en pacientes con lesiones de rodilla. El sistema de control debe aprender a asistir el movimiento
de la pierna del paciente de forma suave y eficiente, minimizando el esfuerzo del motor y maximizando la
fluidez del movimiento. Antes de trabajar con el hardware real, el equipo de ingeniería necesita validar si los métodos de gradiente de política son apropiados para este dominio de control continuo, usando entornos
de simulación estándar como proxy del problema real.
Su grupo ha sido contratado para implementar y comparar REINFORCE con línea base y un Actor-Critic simple,  analizar sus propiedades de convergencia, e investigar el estado del arte en control de exoesqueletos
con RL para producir un dictamen técnico fundamentado.

## **Task 1**

**1. El entorno de simulación que usarán es LunarLanderContinuous-v2 de Gymnasium, que tiene un
espacio de acción continuo de dos dimensiones representando la fuerza de dos propulsores.
Argumenten formalmente por qué Q-Learning tabular y DQN son inapropiados para este entorno. Su
argumento debe mencionar explícitamente el espacio de acción, el operador argmax, y la
representación de la política.**





El aprendizaje Q tabular requiere construir y actualizar una tabla con un valor $Q(s,a)$ para cada combinación de estado y acción, lo cual solo es viable cuando el espacio de acciones es finito y de tamaño manejable. En LunarLanderContinuous-v2 el espacio de acción es continuo y bidimensional, ya que cada componente representa la fuerza aplicada por uno de los dos propulsores dentro de un rango real. Esto implica que existen infinitas acciones posibles para cada estado, de modo que una tabla Q no puede representarlas ni actualizarse de forma completa. Incluso si se discretizara el espacio de acción para forzarlo a ser tabular, la dimensionalidad crece de forma combinatoria con el número de divisiones elegidas por cada dimensión, lo que vuelve la representación impráctica y degrada la resolución del control, algo particularmente grave en un dominio donde se busca un movimiento suave y preciso.

DQN resuelve el problema de representar estados de alta dimensión mediante una red neuronal, pero conserva la misma limitación estructural que Q-Learning tabular en cuanto a la selección de acciones. Tanto para elegir la acción a ejecutar como para calcular el objetivo de entrenamiento mediante bootstrapping, DQN necesita evaluar

$$\arg\max_{a} Q(s,a)$$

Ese operador argmax asume que se puede enumerar o resolver eficientemente sobre un conjunto finito de acciones. En un espacio continuo no existe forma cerrada para resolver ese argmax, y aproximarlo numéricamente dentro del ciclo de control resulta costoso e impreciso, y no es lo que implementa la arquitectura estándar de DQN.

Finalmente, ambos métodos derivan la política de forma implícita y determinista a partir de Q, es decir

$$\pi(s) = \arg\max_{a} Q(s,a)$$

Esa representación no permite expresar una salida continua y graduada como la fuerza de dos propulsores, ni políticas estocásticas suaves útiles para explorar un espacio continuo. Por estas tres razones (espacio de acción continuo e infinito, intratabilidad del operador argmax, y una representación de política implícita y discreta) tanto Q-Learning tabular como DQN resultan inapropiados para este entorno.

**2. Para LunarLanderContinuous-v2, la política se parametrizará como una distribución Gaussiana
𝜋𝜃(𝑎 ∣ 𝑠) = 𝒩(𝜇𝜃, (𝑠)𝜎2𝐼) donde 𝜇𝜃(𝑠) es la salida de una red neuronal. Expliquen cómo se calcula ∇𝜃 ln 𝜋𝜃 (𝐴𝑡 ∣ 𝑆𝑡) para esta parametrización específica. Desarrollen la expresión analítica del gradiente del logaritmo de la densidad Gaussiana respecto a 𝜃, identificando qué parte depende de 𝜃 y qué parte no.**



La política se define como

$$\pi_\theta(a \mid s) = \mathcal{N}(\mu_\theta(s), \sigma^2 I)$$

una Gaussiana multivariante con media dependiente del estado a través de la red neuronal y varianza $\sigma^2 I$, donde $\sigma$ es un parámetro aprendible compartido entre las dimensiones de acción.

La densidad, para una acción de dimensión $d$, es

$$\pi_\theta(a \mid s) = \frac{1}{(2\pi)^{d/2}\sigma^d} \exp\left(-\frac{\lVert a - \mu_\theta(s) \rVert^2}{2\sigma^2}\right)$$

Tomando el logaritmo natural,

$$\ln \pi_\theta(a \mid s) = -\frac{d}{2}\ln(2\pi) - d\ln\sigma - \frac{\lVert a - \mu_\theta(s) \rVert^2}{2\sigma^2}$$

Para obtener $\nabla_\theta \ln \pi_\theta(A_t \mid S_t)$ se deriva esta expresión respecto a los parámetros $\theta$ de la red que produce $\mu_\theta(s)$. Los dos primeros términos, $-\frac{d}{2}\ln(2\pi)$ y $-d\ln\sigma$, no dependen de los parámetros de la red de la media, ya que solo dependen de $\sigma$ y de la constante $d$, por lo que su gradiente respecto a esos parámetros es cero. El único término que depende de $\theta$ es el tercero, y aplicando la regla de la cadena se obtiene

$$\nabla_\theta \ln \pi_\theta(A_t \mid S_t) = \frac{1}{\sigma^2}\left(A_t - \mu_\theta(S_t)\right) \cdot \nabla_\theta \mu_\theta(S_t)$$

Aquí $\nabla_\theta \mu_\theta(S_t)$ es el jacobiano de la salida de la red respecto a sus parámetros, calculado automáticamente por PyTorch mediante autodiferenciación. Este gradiente apunta en la dirección que ajusta los parámetros para que $\mu_\theta(S_t)$ se acerque a la acción efectivamente tomada $A_t$, y su magnitud está escalada por el error de predicción $(A_t - \mu_\theta(S_t))$ y por $1/\sigma^2$. Si $\sigma$ también se trata como parámetro aprendible, se calcula además su propio gradiente

$$\frac{\partial \ln \pi_\theta}{\partial \sigma} = -\frac{d}{\sigma} + \frac{\lVert A_t - \mu_\theta(S_t) \rVert^2}{\sigma^3}$$

que aumenta $\sigma$ cuando el error entre la acción tomada y la media predicha es sistemáticamente grande, y lo reduce en caso contrario, regulando así la exploración.

**3. Comparen formalmente REINFORCE con línea base y Actor-Critic en términos de sesgo y varianza del estimador del gradiente. Para cada algoritmo identifiquen: qué usa como estimador de la ventaja 𝐴̂ 𝑡, qué componente introduce sesgo, y qué componente introduce varianza. Predigan cuál algoritmo
esperan que converja más rápido en LunarLanderContinuous-v2 y justifiquen esa predicción.**



REINFORCE con línea base usa como estimador de la ventaja $\hat{A}_t$ la diferencia

$$\hat{A}_t = G_t - b(S_t)$$

donde $G_t$ es el retorno Monte Carlo obtenido sumando las recompensas reales observadas desde el paso $t$ hasta el final del episodio, y $b(S_t)$ es una red separada entrenada para aproximar $V^\pi(S_t)$. Este estimador es insesgado, porque $G_t$ es una muestra directa del retorno verdadero y la línea base no introduce sesgo, dado que su contribución esperada al gradiente es cero al no depender de la acción tomada. El costo de este estimador es una varianza alta, ya que $G_t$ acumula la aleatoriedad de todas las recompensas y transiciones futuras del episodio completo, lo cual se agrava en episodios largos como los de LunarLanderContinuous-v2.

Actor-Critic reemplaza $G_t$ por el error TD de un paso,

$$\delta_t = R_{t+1} + \gamma \hat{V}_w(S_{t+1}) - \hat{V}_w(S_t)$$

que actúa como estimador de la ventaja. Este estimador introduce sesgo porque depende de $\hat{V}_w$, una aproximación aprendida e imperfecta, sobre todo en las primeras etapas del entrenamiento cuando el Critic aún no ha convergido, ya que el bootstrapping propaga ese error de aproximación hacia la actualización del Actor. A cambio, la varianza se reduce considerablemente porque $\delta_t$ depende solo de una recompensa y una transición, no de la trayectoria completa.

Se espera que Actor-Critic converja más rápido en LunarLanderContinuous-v2. Los episodios de este entorno pueden extenderse varios cientos de pasos con dinámica física ruidosa, lo que hace que los retornos Monte Carlo de REINFORCE tengan varianza muy alta y que las actualizaciones, al ocurrir solo una vez por episodio completo, sean poco frecuentes. Actor-Critic, al actualizar en cada paso con un estimador de menor varianza, aprovecha mejor cada muestra de interacción y suaviza la optimización, lo cual típicamente se traduce en convergencia más rápida y estable, a costa de tolerar cierto sesgo inicial mientras el Critic mejora.

**4. El entorno de exoesqueleto real tiene una restricción que LunarLanderContinuous-v2 no tiene:
las acciones deben ser suaves en el tiempo para no causar movimientos bruscos que dañen al
paciente. Argumenten cómo modificarían la función de recompensa y la parametrización de la
política para incorporar esa restricción. ¿Cambiaría eso la elección entre REINFORCE y Actor-Critic?**



Para incorporar la restricción de suavidad se puede modificar la función de recompensa añadiendo un término de penalización proporcional al cambio entre acciones consecutivas, por ejemplo

$$r'_t = r_t - \lambda \lVert a_t - a_{t-1} \rVert^2$$

de forma que el agente sea penalizado cuando ejecuta variaciones bruscas en la fuerza aplicada al motor. Esta penalización puede extenderse a la segunda diferencia, es decir a la aceleración de la acción, si se busca limitar también cambios abruptos en la tasa de variación y no solo en la variación misma.

En cuanto a la parametrización de la política, el esquema actual depende únicamente del estado actual y no tiene memoria de la acción anterior, lo cual dificulta imponer suavidad de forma explícita. Una modificación razonable es aumentar la entrada de la red con la acción previa, de modo que $\mu_\theta$ dependa del par $(S_t, a_{t-1})$, permitiendo que la red module su salida en función de lo ya ejecutado. Otra alternativa es parametrizar la salida como un incremento acotado respecto a la acción anterior,

$$a_t = a_{t-1} + \Delta_\theta(S_t)$$

limitando explícitamente cuánto puede cambiar la acción en un paso. También es razonable introducir una arquitectura recurrente, como LSTM o GRU, que dé memoria temporal a la política, y sustituir el ruido de exploración Gaussiano independiente por un proceso de ruido correlacionado en el tiempo, como un proceso de Ornstein-Uhlenbeck, para que la exploración misma sea más suave.

Estas modificaciones no alteran la validez del teorema de gradiente de política ni el mecanismo básico de optimización, por lo que en principio tanto REINFORCE como Actor-Critic siguen siendo aplicables. Sin embargo, sí refuerzan la preferencia por Actor-Critic. Al convertir la recompensa en una señal densa que se entrega en cada paso, ya que el término de suavidad se evalúa en cada transición y no solo al final del episodio, el aprendizaje por diferencias temporales que usa Actor-Critic puede aprovechar esa señal inmediata con mayor eficiencia que un estimador Monte Carlo que solo se actualiza al cierre del episodio. Además, la menor varianza de Actor-Critic resulta especialmente valiosa en un dominio de seguridad como la rehabilitación médica, donde actualizaciones erráticas de alta varianza son más difíciles de tolerar que un sesgo moderado y controlable proveniente de un Critic bien entrenado.

**Referencias**

- Hausknecht, M., & Stone, P. (2015). Deep recurrent Q-learning for partially observable MDPs. *AAAI Fall Symposium Series*. https://arxiv.org/abs/1507.06527
- Lillicrap, T. P., Hunt, J. J., Pritzel, A., Heess, N., Erez, T., Tassa, Y., Silver, D., & Wierstra, D. (2015). Continuous control with deep reinforcement learning. *arXiv preprint*. https://arxiv.org/abs/1509.02971
- Mysore, S., Mabsout, B., Mancuso, R., & Saenko, K. (2021). Regularizing action policies for smooth control with reinforcement learning. *2021 IEEE International Conference on Robotics and Automation (ICRA)*. https://arxiv.org/abs/2012.06644